# 自定义 Autograd 与 gradcheck

## 学习目标

使用 `torch.autograd.Function` 实现自定义算子，理解保存中间值、反向输入梯度和数值梯度检查。

## 概念模型

自定义 Function 的 `forward` 负责输出，`backward` 返回每个可微输入的梯度。生产代码优先使用 PyTorch 内置算子；只有确有必要时才自定义。

In [ ]:
import torch
from torch.autograd import Function

class SquarePlusOne(Function):
    @staticmethod
    def forward(ctx, inputs):
        ctx.save_for_backward(inputs)
        return inputs.square() + 1
    @staticmethod
    def backward(ctx, grad_output):
        (inputs,) = ctx.saved_tensors
        return grad_output * 2 * inputs

square_plus_one = SquarePlusOne.apply
x = torch.tensor([2.0], requires_grad=True)
square_plus_one(x).sum().backward()
print('value and gradient:', square_plus_one(x.detach()), x.grad)
assert x.grad.item() == 4

### 实验 1：gradcheck

`gradcheck` 使用 double 精度数值差分检查自定义 backward，适合验证局部梯度实现。

In [ ]:
candidate = torch.randn(3, dtype=torch.double, requires_grad=True)
passed = torch.autograd.gradcheck(square_plus_one, (candidate,), eps=1e-6, atol=1e-4)
print('gradcheck:', passed)
assert passed

## 检查点

说明 `ctx.save_for_backward` 保存什么、`backward` 为什么接收 `grad_output`，以及为什么 gradcheck 通常使用 double。

## 试一试

故意把 backward 中的 `2` 改成 `3`，观察 gradcheck 如何失败；再实现 `abs` 的自定义 backward 并讨论零点不可导。

## 常见错误与调试

返回梯度数量不匹配、忘记处理 broadcast、在 backward 中构建不必要的计算图、使用 float32 导致数值检查不稳定。